# 🚀 Advanced PDF RAG System

## 적용된 고급 기법
1. **Hybrid Search** - 벡터 검색 + BM25 키워드 검색 결합
2. **Reranker** - Cross-Encoder로 검색 결과 재정렬
3. **Multi-Query Retrieval** - LLM이 질문을 다양하게 재작성하여 검색
4. **Contextual Compression** - 검색된 문서에서 관련 부분만 추출
5. **개선된 System Prompt** - 구조화된 답변 및 신뢰도 평가 포함
6. **대화 기록 관리** - 이전 대화 맥락 유지


## 1. 패키지 설치

In [4]:
! uv pip install langchain langchain-community langchain-openai langchain-chroma
! uv pip install pypdf rank-bm25 sentence-transformers gradio python-dotenv

Using Python 3.12.12 environment at: C:\Users\jino7\Desktop\LangChain_Core_26_8_24\.venv
Audited 4 packages in 33ms
Using Python 3.12.12 environment at: C:\Users\jino7\Desktop\LangChain_Core_26_8_24\.venv
Resolved 68 packages in 295ms
 Downloaded transformers
Prepared 4 packages in 4.56s
Uninstalled 1 package in 8ms
Installed 15 packages in 13.70s
 + joblib==1.5.3
 + mpmath==1.3.0
 + narwhals==2.25.0
 + networkx==3.6.1
 + rank-bm25==0.2.2
 + safetensors==0.8.0
 + scikit-learn==1.9.0
 + scipy==1.18.1
 + sentence-transformers==6.0.0
 + setuptools==84.0.0
 + sympy==1.14.0
 + threadpoolctl==3.6.0
 - tokenizers==0.23.1
 + tokenizers==0.22.2
 + torch==2.13.0
 + transformers==5.15.1


## 2. 환경 설정 및 임포트

In [5]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)


True

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors.chain_extract import LLMChainExtractor
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

import gradio as gr

print("✅ 모든 패키지 임포트 완료")

✅ 모든 패키지 임포트 완료


## 3. PDF 로드 및 청킹 (기존 대비 개선)

**개선 포인트:**
- `chunk_size` 축소 (500) → 더 정밀한 검색
- `chunk_overlap` 증가 (100) → 문맥 손실 방지
- 각 청크에 `chunk_id` 메타데이터 추가 → 추적 가능성

In [7]:
PDF_PATH = "test.pdf"

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

for doc in documents:
    doc.metadata["source_file"] = os.path.basename(PDF_PATH)

# ✅ 개선: chunk_size 줄이고 overlap 늘려 문맥 유지
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", "。", ".", " ", ""]  # 한국어 문장 구분자 포함
)

chunks = text_splitter.split_documents(documents)

# ✅ 개선: chunk_id 메타데이터 추가
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i

print(f"📄 총 {len(documents)}페이지 로드")
print(f"🔪 총 {len(chunks)}개 청크로 분할")
print(f"\n[청크 예시]")
print(chunks[2].page_content[:200])

📄 총 11페이지 로드
🔪 총 17개 청크로 분할

[청크 예시]
LGFA PLAYING Rules 2022 2 
 
 
경기장 
1. 여자 게일식 축구는 U-14 경기부터 GAA 풀사이즈 
경기장에서 진행됩니다. U-13 이하 등급의 경우 경기장 
크기가 축소될 수 있습니다. 
2. 15 인제 미만의 경기일 경우 조직 위원회는 경기장의 크기, 
득점 공간, 경기 시간을 단축할 수 있습니다. 
 
선수 
1. 임신 가능성


## 4. 벡터스토어 구축

In [8]:
DB_NAME = "pdf_vector_db_advanced"
MODEL = "gpt-5-mini"  # 또는 "gpt-4o-mini"

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(temperature=0, model_name=MODEL)

# 기존 DB 삭제 후 재생성
if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME
)

print(f"✅ 벡터스토어 생성 완료: {vectorstore._collection.count()}개 문서")

✅ 벡터스토어 생성 완료: 17개 문서


## 5. 🔥 고급 기법 1: Hybrid Search (BM25 + Vector)

- **BM25**: 키워드 기반 검색 → 정확한 단어 매칭에 강함
- **Vector Search**: 의미 기반 검색 → 유사한 의미의 문장에 강함
- **Ensemble**: 두 결과를 가중 평균으로 결합 → 두 방법의 장점 모두 활용

In [9]:
# BM25 리트리버 (키워드 검색)
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 3  # 상위 3개 반환

# Vector 리트리버 (의미 검색)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# ✅ Hybrid: BM25 40% + Vector 60% 가중 결합
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6]
)

# 테스트
test_results = hybrid_retriever.invoke("경기 시간은 얼마나 되나요?")
print(f"✅ Hybrid Search 결과: {len(test_results)}개 문서 검색")
for i, doc in enumerate(test_results):
    print(f"\n[{i+1}] 페이지 {doc.metadata.get('page', '?')+1}: {doc.page_content[:80]}...")

✅ Hybrid Search 결과: 4개 문서 검색

[1] 페이지 5: 착용해야 합니다. 
 
        경기 시간 
1. U-14 이상의 경기 시간은 전반 후반 각각 30 분, 총 
1 시간입니다. 
 
2. ...

[2] 페이지 5: LGFA PLAYING Rules 2022 3 
  
 
2. 여자 게일식 축구 경기 중 모든 선수는 자격을 갖춘 의사나 
치과의사가 따로 지시...

[3] 페이지 7: 3. 경기 중 주심에 의해 퇴장당한 선수는 교체될 수 없습니다. 
그러나 정규 시간에 퇴장 명령을 받은 선수는 연장 시간에는 
교체될 수 있습니...

[4] 페이지 6: LGFA PLAYING Rules 2022 4 
 
 
4. U13 이하의 경기 시간은 조직위원회가 정하는 바에 
따라 축소할 수 있습니다. 
...


## 6. 🔥 고급 기법 2: Multi-Query Retrieval

- LLM이 원래 질문을 **3가지 다른 표현**으로 재작성
- 각 질문으로 독립 검색 후 결과 통합
- 다양한 관점에서 관련 문서를 찾을 수 있어 검색 재현율(Recall) 향상

In [10]:
# Multi-Query를 위한 전용 LLM (질문 재작성용)
query_rewrite_llm = ChatOpenAI(temperature=0.3, model_name=MODEL)

# ✅ Multi-Query Retriever
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=hybrid_retriever,
    llm=query_rewrite_llm
)

# 로깅으로 재작성된 쿼리 확인
import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

test_results = multi_query_retriever.invoke("선수 복장 규정은?")
print(f"\n✅ Multi-Query 결과: {len(test_results)}개 고유 문서 검색")


✅ Multi-Query 결과: 6개 고유 문서 검색


## 7. 🔥 고급 기법 3: Contextual Compression

- 검색된 문서 전체를 LLM에 넘기지 않고, **질문과 관련된 부분만 압축 추출**
- 노이즈 감소 → LLM이 핵심에 집중 → 답변 품질 향상
- 토큰 사용량 절감 (비용 절약)

In [11]:
# ✅ Contextual Compression: LLM이 청크에서 관련 문장만 추출
compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=hybrid_retriever
)

print("✅ Contextual Compression Retriever 설정 완료")
print("ℹ️  실제 질문 시 자동으로 관련 내용만 추출됩니다")

✅ Contextual Compression Retriever 설정 완료
ℹ️  실제 질문 시 자동으로 관련 내용만 추출됩니다


## 8. 🔥 고급 기법 4: 개선된 System Prompt

**기존 프롬프트 문제점:**
- 단순히 '컨텍스트를 활용하라'만 명시
- 답변 형식이 일정하지 않음
- 출처 정보 부실

**개선된 프롬프트:**
- 역할(Role) 명확히 정의
- 답변 구조 강제 (관련도 평가 → 답변 → 출처 → 주의사항)
- 모를 때 솔직히 말하도록 유도
- 한국어 출력 명시

In [12]:
ADVANCED_SYSTEM_PROMPT = """\
당신은 PDF 문서를 기반으로 정확하고 신뢰할 수 있는 답변을 제공하는 전문 문서 분석 어시스턴트입니다.

[역할과 원칙]
- 반드시 제공된 컨텍스트 문서에 근거하여 답변합니다.
- 문서에 없는 내용을 임의로 추측하거나 지어내지 않습니다.
- 확실하지 않은 경우 "문서에서 해당 정보를 찾을 수 없습니다"라고 명확히 말합니다.
- 항상 한국어로 답변합니다.

[답변 형식]
다음 구조로 답변하세요:

**📊 관련도 평가:** (상/중/하) - 검색된 문서가 질문에 얼마나 관련 있는지 한 줄 설명

**💬 답변:**
질문에 대한 핵심 답변을 명확하게 서술하세요. 목록이 필요한 경우 번호나 불릿을 사용하세요.

**📌 출처:**
답변에 사용된 문서 페이지를 명시하세요. (예: 3페이지, 5페이지)

**⚠️ 주의사항:** (해당하는 경우만)
문서의 한계나 추가 확인이 필요한 사항이 있으면 기술하세요.

---
[참고 문서]
{context}
---
"""

print("✅ 개선된 System Prompt 설정 완료")

✅ 개선된 System Prompt 설정 완료


## 9. 최종 RAG 파이프라인 구성

모든 기법을 통합한 `answer_question_advanced` 함수:
1. Multi-Query로 질문 다각화
2. Hybrid Search로 검색
3. 컨텍스트 포맷팅 (페이지 정보 포함)
4. 개선된 프롬프트로 LLM 호출
5. 대화 기록 유지

In [13]:
def format_docs_with_metadata(docs: list) -> str:
    """검색된 문서를 페이지 정보와 함께 포맷팅"""
    formatted = []
    seen = set()
    for doc in docs:
        content = doc.page_content.strip()
        if content in seen:  # 중복 제거
            continue
        seen.add(content)
        page = doc.metadata.get('page', '?')
        page_display = int(page) + 1 if isinstance(page, int) else page
        formatted.append(f"[페이지 {page_display}]\n{content}")
    return "\n\n".join(formatted)


def answer_question_advanced(question: str, history: list) -> str:
    """
    고급 RAG 파이프라인:
    Multi-Query → Hybrid Search → 중복 제거 → 압축 → LLM 답변
    """
    # ✅ Step 1: Multi-Query로 다양한 관점에서 문서 검색
    docs = multi_query_retriever.invoke(question)

    # ✅ Step 2: 컨텍스트 포맷팅 (페이지 메타데이터 포함)
    context = format_docs_with_metadata(docs)

    # ✅ Step 3: 대화 기록을 포함한 메시지 구성
    messages = [SystemMessage(content=ADVANCED_SYSTEM_PROMPT.format(context=context))]

    # 이전 대화 기록 추가 (최근 4턴만 유지 - 토큰 절약)
    for user_msg, bot_msg in history[-4:]:
        if user_msg:
            messages.append(HumanMessage(content=user_msg))
        if bot_msg:
            messages.append(SystemMessage(content=f"이전 답변: {bot_msg}"))

    # 현재 질문 추가
    messages.append(HumanMessage(content=question))

    # ✅ Step 4: LLM 호출
    response = llm.invoke(messages)
    return response.content


# 테스트
print("=" * 60)
print(answer_question_advanced("U-14 이상의 경기 시간은?", []))
print("=" * 60)

📊 관련도 평가: 상 - 문서(소책자)에 해당 규정이 명확히 기재되어 있습니다.

💬 답변:
U-14 이상의 경기 시간은 전·후반 각각 30분으로, 총 경기 시간은 1시간입니다.

📌 출처:
5페이지

⚠️ 주의사항:
정규 시간의 경우 하프타임은 15분을 초과하지 않아야 한다는 규정도 같은 페이지에 명시되어 있습니다.


## 10. Gradio UI 실행

In [16]:
demo = gr.ChatInterface(
    fn=answer_question_advanced,
    title="📄 Advanced PDF RAG Chatbot",
    description=(
        "**고급 RAG 기법 적용:** Hybrid Search (BM25 + Vector) · "
        "Multi-Query Retrieval · Contextual Compression · 개선된 프롬프트"
    ),
    examples=[
        "경기 시간은 얼마나 되나요?",
        "선수 복장 규정을 설명해주세요",
        "교체 선수는 몇 명까지 가능한가요?",
        "출혈 선수는 어떻게 처리하나요?",
    ],
    
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
